

# Notebook Overview
Ths notebook demonstrates the first deployment for the KNN Book Reccomendation
## Conclusion
In this implementation My model possibly failed  because of fewer user dimensions leading to very low distances, to possibly amplify my distance I must use more user dimensions. This will be implmented in the next deployment  

In [ ]:
# import libraries (you may add additional imports but you may not have to)
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/books/book-crossings.zip

!unzip book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

--2026-05-17 18:41:55--  https://cdn.freecodecamp.org/project-data/books/book-crossings.zip
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 104.26.3.33, 104.26.2.33, 172.67.70.149, ...
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|104.26.3.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26085508 (25M) [application/zip]
Saving to: ‘book-crossings.zip’

book-crossings.zip  100%[===================>]  24.88M  2.55MB/s    in 23s     

2026-05-17 18:42:19 (1.06 MB/s) - ‘book-crossings.zip’ saved [26085508/26085508]

Archive:  book-crossings.zip
  inflating: BX-Book-Ratings.csv     
  inflating: BX-Books.csv            
  inflating: BX-Users.csv            


# Original Dataset

In [ ]:
df = pd.read_csv(
    books_filename,
    encoding = "ISO-8859-1",
    sep=";",
    on_bad_lines='skip'
)

/tmp/ipykernel_4082/755777257.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


In [ ]:
df.head()

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...


In [ ]:
# import csv data into dataframes
df_books = pd.read_csv(
    books_filename,
    encoding = "ISO-8859-1", # latin encoder
    sep = ";",
    header = 0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'], # specifies columns to use
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'}) # specifies datatype of columns


In [ ]:
df_books.head()

,isbn,title,author
0,0195153448,Classical Mythology,Mark P. O. Morford
1,0002005018,Clara Callan,Richard Bruce Wright
2,0060973129,Decision in Normandy,Carlo D'Este
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata
4,0393045218,The Mummies of Urumchi,E. J. W. Barber


In [ ]:
df_books.tail()

,isbn,title,author
271374,0440400988,There's a Bat in Bunk Five,Paula Danziger
271375,0525447644,From One to One Hundred,Teri Sloat
271376,006008667X,Lily Dale : The True Story of the Town that Ta...,Christine Wicker
271377,0192126040,Republic (World's Classics),Plato
271378,0767409752,A Guided Tour of Rene Descartes' Meditations o...,Christopher Biffle


In [ ]:
# title to isbn matcher

titles = df_books['title'].tolist()
isbn = df_books['isbn'].tolist()

isbn_title = {code:title for code,title in zip(isbn,titles)}


In [ ]:
len(df_books)

271379

In [ ]:
len(df_books['isbn'].unique()) #all books are unique

271379

In [ ]:
len(isbn)

271379

# Original Ratings Dataset

In [ ]:
df_r = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";")
df_r.columns.tolist()

['User-ID', 'ISBN', 'Book-Rating']

In [ ]:
len(df_r['User-ID'].unique())

105283

In [ ]:
df_ratings = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})

In [ ]:
df_ratings.head(n = 20)

,user,isbn,rating
0,276725,034545104X,0.0
1,276726,0155061224,5.0
2,276727,0446520802,0.0
3,276729,052165615X,3.0
4,276729,0521795028,6.0
5,276733,2080674722,0.0
6,276736,3257224281,8.0
7,276737,0600570967,6.0
8,276744,038550120X,7.0
9,276745,342310538,10.0


In [ ]:
# Counts occurrences of every user
# same users make ratings on different books
users_counts = df_ratings["user"].value_counts()
print(users_counts[:])

user
11676     13602
198711     7550
153662     6109
98391      5891
35859      5850
          ...  
69281         1
69239         1
69241         1
69245         1
276733        1
Name: count, Length: 105283, dtype: int64


In [ ]:
df_ratings.tail(n=3)

,user,isbn,rating
1149777,276709,0515107662,10.0
1149778,276721,0590442449,10.0
1149779,276723,05162443314,8.0


In [ ]:
book_title.get('0155061224',0)

'Rites of Passage'

In [ ]:
len(df_ratings)

1149780

In [ ]:
len(df_ratings['isbn'].unique())

340556

# Addressing Inconsistencies

The Ratings have more unique codes than observed in the original book filenames this means  ```340556 - 271379 = 69177``` will have missing titles

### Checking for Subset Relationship

In [ ]:
# Convert unique ISBNs to sets
isbns_in_df_books = set(df_books['isbn'])
isbns_in_df_ratings = set(df_ratings['isbn'])

# Checks if all ISBNs in df_books are present in df_ratings
is_books_subset_of_ratings = isbns_in_df_books.issubset(isbns_in_df_ratings)

print(f"Are all ISBNs from df_books present in df_ratings? {is_books_subset_of_ratings}")


is_ratings_subset_of_books = isbns_in_df_ratings.issubset(isbns_in_df_books)
print(f"Are all ISBNs from df_ratings present in df_books? {is_ratings_subset_of_books}")

if not is_books_subset_of_ratings:
    missing_in_ratings = isbns_in_df_books - isbns_in_df_ratings
    print(f"\nNumber of ISBNs in df_books NOT found in df_ratings: {len(missing_in_ratings)}")

if not is_ratings_subset_of_books:
    missing_in_books = isbns_in_df_ratings - isbns_in_df_books
    print(f"Number of ISBNs in df_ratings NOT found in df_books: {len(missing_in_books)}")

Are all ISBNs from df_books present in df_ratings? False
Are all ISBNs from df_ratings present in df_books? False

Number of ISBNs in df_books NOT found in df_ratings: 1209
Number of ISBNs in df_ratings NOT found in df_books: 70386


# Infrence


```Number of ISBNs in df_books NOT found in df_ratings: 1209``` - This shows that they are books that exist by different authors but lack ratings. Hence can't compare to anything for getting right reccommendation

```Number of ISBNs in df_ratings NOT found in df_books: 70386``` - This shows that they are 70386 books that have been rated by users but dont have no record in books meaning status remains unknown which is not desired when making reccomendations. This can be users who made entries either misspelled the isbn code or didnt leave details about those specific books i.e books can't be classified if they miss metadata.

In [ ]:
# unique users
len(df_ratings['user'].unique())

105283

In [ ]:
# users

user_ids = sorted(df_ratings['user'].unique().tolist())
print(user_ids[:30])
len(user_ids)
print(user_ids[-1])


[2, 7, 8, 9, 10, 12, 14, 16, 17, 19, 20, 22, 23, 26, 32, 36, 38, 39, 42, 44, 51, 53, 56, 64, 67, 68, 69, 70, 73, 75]
278854


In [ ]:
len(user_ids)

105283

In [ ]:
u = 278854
user_indices = df_ratings[df_ratings['user'] == u].index
print(f"Row numbers (indices) where user {u} occurs: {user_indices.tolist()}")


Row numbers (indices) where user 278854 occurs: [9553, 9554, 9555, 9556, 9557, 9558, 9559, 9560]


In [ ]:
isbn_indices = df

In [ ]:
# book ids as recorded in the df_book

isbn.sort()
print(isbn[:20])

['0000913154', '0001010565', '0001046438', '0001046713', '000104687X', '0001046934', '0001047213', '0001047647', '0001047663', '0001047868', '0001047973', '000104799X', '0001048082', '0001048473', '0001049879', '0001052039', '0001053736', '0001053744', '0001055607', '0001056107']


# pivotting Dataframe

This approach tilts my dataframe in such a way the column isbn in df_ratings gets unique isbn codes and columns get unique user with the ratings




In [ ]:
len(df_ratings)

1149780

In [ ]:
book_counts = df_ratings['isbn'].value_counts()
valid_books = book_counts[book_counts >= 100].index
len(valid_books)

731

In [ ]:
# filtering columns
# only valid isbn codes from the df_books should be included to avoid lack of missing book information
# only books with more than 100 ratings meaning it appears more than 100 times
# only users with 200+ ratings



filtered_df = df_ratings[df_ratings['isbn'].isin(df_books['isbn'])] #drops rows with invalid isbn
book_counts = filtered_df['isbn'].value_counts() #returns frequencies of unique books
valid_books = book_counts[book_counts >= 100].index

In [ ]:
len(valid_books)

727

In [ ]:
filtered_df.shape

(1031175, 3)

In [ ]:
df_ratings.shape

(1149780, 3)

In [ ]:
print(valid_books[:5])

Index(['0971880107', '0316666343', '0385504209', '0060928336', '0312195516'], dtype='object', name='isbn')


In [ ]:
# dropping users
filtered_df = filtered_df[filtered_df['isbn'].isin(valid_books)]



In [ ]:
user_counts = filtered_df['user'].value_counts()
valid_users = user_counts[user_counts >= 200].index

In [ ]:
len(valid_users)

20

In [ ]:
# dropping users and books

#filtered_df = filtered_df[filtered_df['isbn'].isin(valid_books)]
filtered_df = filtered_df[filtered_df['user'].isin(valid_users)]
filtered_df.shape

(5368, 3)

In [ ]:
254 in filtered_df['user']

False

In [ ]:
len(filtered_df['isbn'].unique())

727

In [ ]:
#Over 1 million entries were dropped
len(df_ratings) - len(filtered_df)

1144412

In [ ]:
filtered_df.head()

,user,isbn,rating
45462,11676,002542730X,6.0
45489,11676,0060008032,8.0
45516,11676,0060096195,0.0
45530,11676,006016848X,9.0
45537,11676,0060173289,0.0


In [ ]:
# conversion to categorical columns

filtered_df['isbn'] = filtered_df['isbn'].astype('category')
filtered_df['user'] = filtered_df['user'].astype('category')

# creating sparse representations, avoids memory usage
#mask_zero = pd.SparseDtype(np.float64, 0)
#filtered_df['rating'] = filtered_df['rating'].astype(mask_zero)


In [ ]:
# pivotting transforms chosen unique column values to rows, columns and rating
pivot_df = filtered_df.pivot_table(index ='isbn',columns = 'user', values = 'rating',fill_value = 0,observed=True)

In [ ]:
len(filtered_df[filtered_df['isbn'] == '002542730X'])

9

In [ ]:
pivot_df.head()

user,11676,16795,21014,23768,35859,43246,52584,55492,60244,76352,78783,102967,135149,153662,185233,198711,204864,230522,232131,238120
isbn,,,,,,,,,,,,,,,,,,,,
002542730X,6.0,0.0,0.0,0.0,0.0,0.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
0060008032,8.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
0060096195,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
006016848X,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
0060173289,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
pivot_df.columns.dtype

CategoricalDtype(categories=[ 11676,  16795,  21014,  23768,  35859,  43246,  52584,
                   55492,  60244,  76352,  78783, 102967, 135149, 153662,
                  185233, 198711, 204864, 230522, 232131, 238120],
, ordered=False, categories_dtype=int32)

In [ ]:
# memory consumption
print(pivot_df.memory_usage(deep = True).sum())

119059


In [ ]:
sparse_matrix = csr_matrix(pivot_df.values)

# creating KNN model

In [ ]:
'''
brute algorithm for comparing a query to each unique point
'''

from sklearn.neighbors import NearestNeighbors
knn_model = NearestNeighbors(metric='cosine', algorithm='brute')
knn_model.fit(sparse_matrix)

NearestNeighbors(algorithm='brute', metric='cosine')

In [ ]:
# getting relevant titles

df = df_books[df_books['isbn'].isin(pivot_df.index)]
df.shape

(727, 3)

# comparing similarities

In [ ]:
set(df['isbn']).issubset(set(pivot_df.index))

True

In [ ]:
# title to isbn dictionary

titles = df['title'].tolist()
isbn = df['isbn'].tolist()

isbn_title = {code:title for code,title in zip(isbn,titles)}

title_isbn = {title:code for code,title in isbn_title.items()}

In [ ]:
# testing finding row positions
# takes a book code returns its row position
pivot_df.index.get_loc('006016848X')

print(sparse_matrix[0]) #returns non zero positions hence for saving computational resources by computing distance

# testing finding book codes once given position

pivot_df.index[3]

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 2 stored elements and shape (1, 20)>
  Coords	Values
  (0, 0)	6.0
  (0, 6)	10.0


'006016848X'

In [ ]:
from sklearn import neighbors

## function to return recommended books - this will be tested
def get_recommends(book):

 #check if title is valid
  if book not in title_isbn:
    print("Enter valid title")
    return

  # create the first dimension
  recommended_books = [book]

  # second dimension

  neighbors = []

  # get book_code

  book_code = title_isbn[book]

  # get row index for book code

  row_index = pivot_df.index.get_loc(book_code)

  # get csr vector i.e where the book is located

  position_vector = sparse_matrix[row_index]

  # find neighbors

  distances, indices = knn_model.kneighbors(position_vector, n_neighbors= 5)


  # flatten distances and indices from 2D to 1D

  distances, indices = distances.flatten(), indices.flatten()


  for index,row in enumerate(indices):

    # get book codes of neighbours

    n_code = pivot_df.index[row]

    # neighbors titles

    n_title = isbn_title[n_code]

    # append to second dimension along with their distances as a 3rd dimension(list)

    neighbors.append([n_title, float(distances[index])])


  # append to main title

  recommended_books.append(neighbors)

  return recommended_books

# Testing function

input - ```get_recommends("The Queen of the Damned (Vampire Chronicles (Paperback))")```

expected_output - ```[
  'The Queen of the Damned (Vampire Chronicles (Paperback))',
  [
    ['Catch 22', 0.793983519077301],
    ['The Witching Hour (Lives of the Mayfair Witches)', 0.7448656558990479],
    ['Interview with the Vampire', 0.7345068454742432],
    ['The Tale of the Body Thief (Vampire Chronicles (Paperback))', 0.5376338362693787],
    ['The Vampire Lestat (Vampire Chronicles, Book II)', 0.5178412199020386]
  ]
]```

In [ ]:
get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")

["Where the Heart Is (Oprah's Book Club (Paperback))",
 [["Where the Heart Is (Oprah's Book Club (Paperback))", 0.0],
  ['Chosen Prey', 0.11811447143554688],
  ['The Weight of Water', 0.1460479497909546],
  ['Insomnia', 0.14954054355621338],
  ['Hannibal', 0.17240118980407715]]]

In [ ]:
title_isbn["The Queen of the Damned (Vampire Chronicles (Paperback))"]

'0345351525'

In [ ]:
pivot_df.index.get_loc('0345351525')

136

In [ ]:
sparse_matrix[136]

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 0 stored elements and shape (1, 20)>

In [ ]:
title_isbn.get('I Know This Much Is True',0)


'0060987561'

In [ ]:
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(2):
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge! 🎉🎉🎉🎉🎉")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()

["Where the Heart Is (Oprah's Book Club (Paperback))", [["Where the Heart Is (Oprah's Book Club (Paperback))", 0.0], ['Chosen Prey', 0.11811447143554688], ['The Weight of Water', 0.1460479497909546], ['Insomnia', 0.14954054355621338], ['Hannibal', 0.17240118980407715]]]
You haven't passed yet. Keep trying!
